In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [19]:
import csv
import math

col_list_map = {
    "avail_balance": {"dtype": "float64"},
    "time": 'drop',
    "eventId_DV_INTENRAL": 'drop',
    "hist_seq": {"dtype": "int64"},
    "sender_available_balance": {"dtype": "float64"},
    "sender_ledger_balance": {"dtype": "float64"},
    "plaid_account_balance": {"dtype": "float64"},
    "amount": {"dtype": "float64"},
    "salary": {"dtype": "float64"},
    "ssn_tax_number": {"dtype": "int64"},
    "mobile_phone_number": {"dtype": "int64"},
    "current_sofi_employee": {"dtype": "int64"},
    "years_of_experience": {"dtype": "int64"},
    "receiver_customer_status": {"dtype": "int64"},
    "receiver_phone_number": {"dtype": "int64"},
    "primary_email_verified": {"dtype": "int64"},
    "mobile_phone_area_code": {"dtype": "int64"},
    "receiver_account_number": {"dtype": "int64"},
    "new_two_factor_data": {"dtype": "int64"},
    "tran_seq": {"dtype": "int64"},
    
    # "user_active": {"dtype": "int64"},
    # Add more mappings as required, following the rules above.
}

input_data_path = './data'
output_data_path = './data/processed'

os.makedirs(output_data_path, exist_ok=True)

def is_csv_valid(filepath):
    """
    Perform basic csv validation for:
        - wrong delimiter
        - broken quoting
        - very long field/malformed line
        - embedded newlines inside quoted text
        - mixed encoding/corrupt file
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as fin:
            # use comma delimiter and strict quoting
            sniff = csv.Sniffer().sniff(fin.read(2048))
            fin.seek(0)
            if sniff.delimiter not in [',']:
                print(f"[ERROR] File '{filepath}' wrong delimiter detected: {repr(sniff.delimiter)}")
                return False
            reader = csv.reader(fin, delimiter=',', quotechar='"', strict=True)
            for i, row in enumerate(reader):
                # Arbitrary long field: warning for fields > 1M chars
                if any(len(field) > 1_000_000 for field in row):
                    print(f"[ERROR] File '{filepath}' very long field detected at row {i}")
                    return False
                if len(row) == 0:
                    print(f"[ERROR] File '{filepath}' empty/malformed line at row {i}")
                    return False
                # Can also check for excessive length, broken quoting, etc.
    except csv.Error as e:
        print(f"[ERROR] File '{filepath}' broken quoting or embedded newline: {e}")
        return False
    except UnicodeDecodeError as e:
        print(f"[ERROR] File '{filepath}' encoding/decoding error: {e}")
        return False
    except Exception as e:
        print(f"[ERROR] File '{filepath}' general error: {e}")
        return False
    return True

for file in os.listdir(input_data_path):
    if file.endswith('.csv'):
        full_input_path = os.path.join(input_data_path, file)
        if not is_csv_valid(full_input_path):
            print(f"[WARN] Skipping invalid CSV file '{file}'")
            continue

        # Load using correct delimiter, quoting, encoding handling
        try:
            df = pd.read_csv(
                full_input_path, 
                delimiter=',', 
                engine='python',  # better at handling malformed lines
                quoting=csv.QUOTE_MINIMAL,
                encoding='utf-8',
                on_bad_lines='warn'  # pandas 1.3+: warns but doesn't crash on malformed lines
            )
        except Exception as e:
            print(f"[ERROR] Could not load '{file}' due to: {e}")
            continue
        
        print(df.head())

        for col, props in col_list_map.items():
            if col in df.columns:
                if props == 'drop':
                    df = df.drop(columns=[col])
                    print(f"[INFO] Dropped column '{col}'")
                elif isinstance(props, dict) and 'dtype' in props:
                    try:
                        old_dtype = df[col].dtype
                        # Convert to numeric, handle commas, and handle NaN for int specifically
                        if props['dtype'] == "float64":
                            # Remove commas, convert to float (NaN are preserved)
                            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '', regex=False), errors='coerce')
                            df[col] = df[col].astype("float64")
                        elif props['dtype'] == "int64":
                            # Remove commas, convert to float first (to allow nan), then to Int64 (nullable int)
                            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '', regex=False), errors='coerce')
                            # Use Pandas nullable Int64 column, so NaNs remain
                            try:
                                df[col] = df[col].astype("Int64")  
                            except Exception as e2:
                                print(f"[WARN] Failed conversion to nullable Int64 for column '{col}': {e2}. Trying to fillna(0) and cast to int64.")
                                df[col] = df[col].fillna(0).astype("int64")
                        else:
                            df[col] = df[col].astype(props['dtype'])
                        print(f"[INFO] Converted column '{col}' from {old_dtype} to {props['dtype']}")
                    except Exception as e:
                        print(f"[ERROR] Failed to convert column '{col}': {e}")

        # Save the processed DataFrame to the /data/processed directory with the same filename
        output_file_path = os.path.join(output_data_path, file)
        try:
            df.to_csv(output_file_path, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
            print(f"[INFO] Saved processed file to {output_file_path}")
        except Exception as e:
            print(f"[ERROR] Failed to save '{output_file_path}': {e}")

   is_restriction_removed               time   user_id  is_restriction_added  \
0                       0  1,752,071,228,000  10004301                     1   
1                       0  1,752,073,051,000  10004301                     1   
2                       0  1,751,535,252,000   1005899                     1   
3                       0  1,752,312,473,000  10081403                     1   
4                       0  1,751,998,091,000  10093253                     1   

                      event_type  \
0  profile_customer_restrictions   
1  profile_customer_restrictions   
2  profile_customer_restrictions   
3  profile_customer_restrictions   
4  profile_customer_restrictions   

                                            event_id  \
0  10004301-profile_customer_restrictions-53-2025...   
1  10004301-profile_customer_restrictions-56-2025...   
2  1005899-profile_customer_restrictions-32-2025-...   
3  10081403-profile_customer_restrictions-53-2025...   
4  10093253-profile_cu